In [3]:

import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')


# ======================================================
# 1. DATA UNDERSTANDING
# ======================================================
import pandas as pd

reviews = [
"I loved this movie, it was amazing and inspiring",
"This is the worst product I have ever bought",
"The service was okay, nothing special",
"Absolutely fantastic experience, highly recommend",
"I am very disappointed with the quality",
"It works fine but has some minor issues",
"Great value for money and excellent performance",
"Terrible customer support, not satisfied",
"The product is average, not too good",
"Superb quality and fast delivery, very happy",
"Waste of money, very bad experience",
"Not bad, but could be improved",
"I really enjoyed using this app",
"Very poor performance and lots of bugs",
"It is okay for the price",
"Amazing design and smooth functionality",
"I hate this item, completely useless",
"Nothing extraordinary, just average",
"Best purchase I have made this year",
"Extremely bad quality and slow response",
"Decent product with acceptable features",
"Loved the interface and user experience",
"This app crashes frequently, very annoying",
"Average experience, neither good nor bad",
"Highly satisfied with the purchase",
"Bad packaging and delayed delivery",
"It performs as expected",
"Excellent product, will buy again",
"Not worth the price at all",
"Okayish performance, not impressive"
]

sentiments = [
"positive",
"negative",
"neutral",
"positive",
"negative",
"neutral",
"positive",
"negative",
"neutral",
"positive",
"negative",
"neutral",
"positive",
"negative",
"neutral",
"positive",
"negative",
"neutral",
"positive",
"negative",
"neutral",
"positive",
"negative",
"neutral",
"positive",
"negative",
"neutral",
"positive",
"negative",
"neutral"
]

df = pd.DataFrame({
    "review": reviews,
    "sentiment": sentiments
})

  # Example: IMDb/Amazon dataset

print("\n========== DATA OVERVIEW ==========\n")
print("Shape:", df.shape)
print("\nColumns:", df.columns)

print("\nSample Data:")
print(df.head())

print("\nClass Distribution:")
print(df['sentiment'].value_counts())


# ======================================================
# 2. NLP PREPROCESSING
# ======================================================

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()


def clean_review(text):
    """Complete preprocessing pipeline"""

    if not isinstance(text, str):
        return ""

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Tokenization
    words = text.split()

    # Remove stopwords + stemming
    processed = []
    for word in words:
        if word not in stop_words:
            processed.append(stemmer.stem(word))

    return " ".join(processed)


print("\n========== PREPROCESSING ==========\n")

df['clean_text'] = df['review'].apply(clean_review)

print(df[['review', 'clean_text']].head())


# ======================================================
# 3. FEATURE ENGINEERING
# ======================================================

print("\n========== FEATURE ENGINEERING ==========\n")

# Bag of Words
bow_vectorizer = CountVectorizer(max_features=5000)
X_bow = bow_vectorizer.fit_transform(df['clean_text'])

# TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf_vectorizer.fit_transform(df['clean_text'])

# Target
y = df['sentiment']


# =======================
# TRAIN-TEST SPLIT
# ======================

X_train_bow, X_test_bow, y_train, y_test = train_test_split(
    X_bow, y, test_size=0.2, random_state=42
)

X_train_tfidf, X_test_tfidf, _, _ = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)


# ================================
# 4. MODEL TRAINING FUNCTION
# ================================

def train_and_evaluate(model, X_train, X_test, y_train, y_test, name):
    """Train model and print evaluation metrics"""

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average='weighted')
    rec = recall_score(y_test, preds, average='weighted')
    f1 = f1_score(y_test, preds, average='weighted')

    print(f"\n{name} Results:")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")

    return acc, prec, rec, f1


# =====================================
# 5. MODEL BUILDING & EVALUATION
# ======================================

print("\n========== MODEL TRAINING (BoW) ==========\n")

models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier()
}

results = {}

for model_name, model_obj in models.items():
    results[f"{model_name}_BoW"] = train_and_evaluate(
        model_obj, X_train_bow, X_test_bow, y_train, y_test, model_name + " (BoW)"
    )


print("\n========== MODEL TRAINING (TF-IDF) ==========\n")

for model_name, model_obj in models.items():
    results[f"{model_name}_TFIDF"] = train_and_evaluate(
        model_obj, X_train_tfidf, X_test_tfidf, y_train, y_test, model_name + " (TF-IDF)"
    )


# ======================================================
# 6. COMPARISON & INSIGHTS
# ======================================================

print("\n========== FINAL COMPARISON ==========\n")

for k, v in results.items():
    print(f"{k}: Accuracy={v[0]:.4f}, F1={v[3]:.4f}")


print("\n========== INSIGHTS ==========\n")

print("""
1. TF-IDF usually performs better than BoW because it reduces importance of frequent words.
2. Logistic Regression often gives stable and high accuracy.
3. Naive Bayes is fast and works well with text data.
4. Decision Tree may overfit on high-dimensional sparse data.
5. Preprocessing (stopword removal + stemming) improves performance significantly.
""")


# ==============================
# OPTIONAL: TEST CUSTOM INPUT
# ==============================

def predict_sentiment(text, vectorizer, model):
    processed = clean_review(text)
    vec = vectorizer.transform([processed])
    prediction = model.predict(vec)
    return prediction[0]


print("\n========== SAMPLE PREDICTION ==========\n")

sample = "This movie was absolutely amazing and fantastic!"
print("Input:", sample)
print("Prediction:", predict_sentiment(sample, tfidf_vectorizer, model_obj))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average


========== DATA OVERVIEW ==========

Shape: (30, 2)

Columns: Index(['review', 'sentiment'], dtype='object')

Sample Data:
                                              review sentiment
0   I loved this movie, it was amazing and inspiring  positive
1       This is the worst product I have ever bought  negative
2              The service was okay, nothing special   neutral
3  Absolutely fantastic experience, highly recommend  positive
4            I am very disappointed with the quality  negative

Class Distribution:
sentiment
positive    10
negative    10
neutral     10
Name: count, dtype: int64

========== PREPROCESSING ==========

                                              review  \
0   I loved this movie, it was amazing and inspiring   
1       This is the worst product I have ever bought   
2              The service was okay, nothing special   
3  Absolutely fantastic experience, highly recommend   
4            I am very disappointed with the quality   

                     